# 🎯 3. Agent Loops and Core Architecture

In notebooks 01-02 we learned what agents are and how LLMs serve as their brain. Now we **build** our first agents.

We will cover:

1. **The minimal agent loop** — a while-loop with LLM + tools
2. **State management** — tracking agent state with TypedDict
3. **Stopping conditions** — how and when the agent finishes
4. **LangGraph StateGraph** — the professional way to build agent graphs
5. **Conditional edges** — dynamic routing based on agent decisions
6. **From scratch to framework** — understanding what LangGraph abstracts

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"

print(f'Model: {LLM_MODEL}')
print(f'API key configured: {bool(os.getenv("OPENAI_API_KEY"))}')

Model: gpt-4o-mini
API key configured: True


## 3.1 Building an Agent Loop from Scratch

Before using any framework, let's build the agent pattern from first principles. This is the pattern that **every** agent framework implements under the hood.

The core idea:

<img src="images/agent-loop-1.png" width="70%" style="border-radius:10px;margin:12px 0;"/>

In [2]:
from openai import OpenAI
import json

client = OpenAI()

# ── Define Tools ──
def get_weather(city: str) -> str:
    """Simulated weather API."""
    weather_data = {
        'london': 'Cloudy, 12°C, 60% humidity',
        'tokyo': 'Sunny, 24°C, 45% humidity',
        'new york': 'Rainy, 8°C, 85% humidity',
        'paris': 'Partly cloudy, 16°C, 55% humidity',
    }
    return weather_data.get(city.lower(), f'No data for {city}')

def calculate(expression: str) -> str:
    """Safe math evaluator."""
    allowed = set('0123456789+-*/.() ')
    if all(c in allowed for c in expression):
        return str(eval(expression))
    return 'Error: invalid expression'

# Tool registry — maps names to functions
TOOL_REGISTRY = {
    'get_weather': get_weather,
    'calculate': calculate,
}

# Tool schemas for OpenAI
TOOL_SCHEMAS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': 'Get current weather for a city',
            'parameters': {
                'type': 'object',
                'properties': {'city': {'type': 'string', 'description': 'City name'}},
                'required': ['city']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'calculate',
            'description': 'Evaluate a math expression',
            'parameters': {
                'type': 'object',
                'properties': {'expression': {'type': 'string', 'description': 'Math expression'}},
                'required': ['expression']
            }
        }
    }
]

print(f'Registered {len(TOOL_REGISTRY)} tools: {list(TOOL_REGISTRY.keys())}')

Registered 2 tools: ['get_weather', 'calculate']


In [3]:
# Όλο το χειροκίνητο run_agent() loop (με thinking, acting, observing,
# tool dispatch και stopping condition) είναι ακριβώς αυτό που
# υλοποιεί εσωτερικά η create_agent. Δείτε εδώ το ίδιο
# αποτέλεσμα σε 3 γραμμές:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

@tool
def get_weather_tool(city: str) -> str:
    """Get current weather for a city."""
    return get_weather(city)

@tool
def calculate_tool(expression: str) -> str:
    """Safely evaluate a math expression."""
    return calculate(expression)

agent_react = create_agent(
    model=ChatOpenAI(model=LLM_MODEL, temperature=0),
    tools=[get_weather_tool, calculate_tool],
    system_prompt='You are a helpful assistant with access to tools. Use tools when needed.',
)

out = agent_react.invoke({'messages': [HumanMessage(content='What is the weather in london?')]})
print(out['messages'][-1].content)


c:\Users\user\Desktop\ai-agents-course-me\venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


The weather in London is currently cloudy, with a temperature of 12°C and 60% humidity.


In [4]:
# Test the factory agent with a multi-step query
out = agent_react.invoke({
    'messages': [HumanMessage(content='What is the weather in Tokyo, and what is 23 * 47?')]
})
for m in out['messages']:
    role = m.__class__.__name__
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'  → {tc["name"]}({tc["args"]})')
    elif getattr(m, 'content', None):
        print(f'{role}: {m.content}')


HumanMessage: What is the weather in Tokyo, and what is 23 * 47?
  → get_weather_tool({'city': 'Tokyo'})
  → calculate_tool({'expression': '23 * 47'})
ToolMessage: Sunny, 24°C, 45% humidity
ToolMessage: 1081
AIMessage: The weather in Tokyo is sunny with a temperature of 24°C and 45% humidity. Additionally, the result of \( 23 \times 47 \) is 1081.


## 3.2 State Management with TypedDict

Our from-scratch agent used a simple `messages` list. In real applications, agents need richer state:

- **Messages**: the conversation history
- **Current step**: where we are in the workflow
- **Intermediate results**: tool outputs, computed values
- **Metadata**: timestamps, token counts, error flags

Python's `TypedDict` gives us type-safe state management:

In [5]:
from typing_extensions import TypedDict, Annotated
import operator

# Define agent state with TypedDict
class AgentState(TypedDict):
    """State that the agent maintains across steps."""
    messages: Annotated[list, operator.add]  # Append-only message list
    current_step: str                        # Which node is executing
    tool_calls_made: int                     # How many tools we've called
    is_complete: bool                        # Whether the agent is done

# Initialize state
initial_state: AgentState = {
    'messages': [{'role': 'user', 'content': 'Hello!'}],
    'current_step': 'start',
    'tool_calls_made': 0,
    'is_complete': False,
}

print(f'State keys: {list(initial_state.keys())}')
print(f'Messages: {len(initial_state["messages"])}')
print(f'Step: {initial_state["current_step"]}')

State keys: ['messages', 'current_step', 'tool_calls_made', 'is_complete']
Messages: 1
Step: start


## 3.3 Introduction to LangGraph

**LangGraph** is the production framework for building agent workflows as **directed graphs**.

Core concepts:
- **StateGraph**: The graph definition — nodes and edges
- **Nodes**: Functions that process state (the steps)
- **Edges**: Connections between nodes (the flow)
- **Conditional Edges**: Dynamic routing based on state
- **START / END**: Special nodes marking entry and exit


<img src="images/intro-to-langgraph.png" width="90%" style="border-radius:10px;margin:12px 0;"/>

<img src="images/start-analyse-gen-end.png" width="90%" style="border-radius:10px;margin:12px 0;"/>

## 3.4 Conditional Edges — Dynamic Routing

Real agents don't follow a fixed path. They make **decisions** at each step. LangGraph supports this with **conditional edges** — functions that return the next node based on state.

<img src="images/conditional-edges-dynamic-routing.png" width="90%" style="border-radius:10px;margin:12px 0;"/>

## 💡 Exercise 3: Build a Support Ticket Router

**Task**: Build a LangGraph that:
1. Takes a customer message
2. Classifies it (billing, technical, feedback)
3. Routes to the appropriate handler
4. Each handler generates a tailored response

Hints:
- Define a `TicketState` with `messages`, `category`, and `response`
- Create a `classify` node and three handler nodes
- Use `add_conditional_edges` for routing

In [12]:
# Exercise 3: Build your Support Ticket Router
# YOUR CODE HERE

# class TicketState(TypedDict):
#     ...

# def classify_ticket(state):
#     ...

# ticket_graph = StateGraph(TicketState)
# ...


## 📝 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Agent Loop** | `while not done: think → act → observe` — the universal agent pattern |
| **State** | TypedDict with `Annotated[list, operator.add]` for message accumulation |
| **LangGraph** | StateGraph + nodes + edges = production-grade agent workflows |
| **Conditional Edges** | `add_conditional_edges(node, router_fn)` for dynamic routing |
| **From Scratch → Framework** | Our raw loop and LangGraph implement the same pattern |

### What's Next

In **Notebook 04: Tools and Function Calling**, we dive deep into the **tools** that give agents their power — function schemas, tool registries, LangChain's `@tool` decorator, and LangGraph's ToolNode.